# 第5回 指数平滑法と回帰

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：予測に、自分の過去と、ほかの系列を、どう使えばよいか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "fable", "feasts", "fpp3"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)
library(fable)
library(ggtime)   # ACFと予測の図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

### A-1. 単純指数平滑法

第2回のベンチマークを思い出します。平均法は、過去の値をすべて同じ重みで使います。ナイーブ法は、最後の値だけを使います。
指数平滑法（exponential smoothing）は、そのあいだです。新しい値ほど重く、古い値ほど軽く使います。

人工の系列を作ります。ゆっくり動く水準（level）に、ノイズが乗っています。

- 水準：$\mu_t = \mu_{t-1} + \eta_t$（ランダムウォーク。$\eta_t$ の標準偏差は0.5）
- 観測：$y_t = \mu_t + \varepsilon_t$（$\varepsilon_t$ の標準偏差は2）

In [ ]:
set.seed(1)
n <- 200
mu <- 50 + cumsum(rnorm(n, sd = 0.5))   # 水準：ゆっくり動く
y  <- mu + rnorm(n, sd = 2)             # 観測：水準 ＋ ノイズ
sim <- tsibble(t = 1:n, y = y, mu = mu, index = t)
ggplot(sim, aes(t)) +
  geom_line(aes(y = y), colour = "grey50") +
  geom_line(aes(y = mu), colour = "red") +
  labs(x = "Time t (step)", y = "Value (grey: observed y, red: level mu)")

見えるのは灰色の線だけです。赤い線（水準）は見えません。

単純指数平滑法（simple exponential smoothing, SES）は、水準の推定値 $\ell_t$ を、1時点ずつ更新します。

$$
\ell_t = \ell_{t-1} + \alpha \left( y_t - \ell_{t-1} \right)
$$

$y_t - \ell_{t-1}$ は、1時点先の予測の外れです。その $\alpha$ 倍だけ、水準を直します。$\alpha$（$0 \le \alpha \le 1$）を平滑化パラメータ（smoothing parameter）と呼びます。

In [ ]:
# 単純指数平滑法：水準を1時点ずつ更新する
ses <- function(y, alpha, l0 = y[1]) {
  n <- length(y)
  fitted <- numeric(n)   # 1時点先の予測（y_t を見る前の水準）
  level  <- numeric(n)   # y_t を見て更新した水準
  prev <- l0
  for (t in 1:n) {
    fitted[t] <- prev
    level[t]  <- prev + alpha * (y[t] - prev)   # 予測の外れの alpha 倍だけ直す
    prev <- level[t]
  }
  data.frame(t = 1:n, y = y, fitted = fitted, level = level)
}
# 1時点先の予測の、2乗誤差の和
sse <- function(alpha, y) sum((y - ses(y, alpha)$fitted)^2)

$\alpha$ を3通りに変えて、水準の推定値を描きます。

In [ ]:
lv <- do.call(rbind, lapply(c(0.02, 0.2, 0.9), function(a)
  cbind(ses(sim$y, a), alpha = sprintf("alpha = %.2f", a))))
ggplot(lv, aes(t)) +
  geom_line(aes(y = y), colour = "grey70") +
  geom_line(aes(y = level), colour = "blue") +
  facet_wrap(~alpha, ncol = 1) +
  labs(x = "Time t (step)", y = "Value (grey: observed y, blue: estimated level)")

- $\alpha$ が小さい：ほとんど動かない。平均法に近い。
- $\alpha$ が大きい：観測を追いかける。ナイーブ法に近い。

$\alpha$ は、1時点先の予測の2乗誤差の和が最も小さくなるように選びます。
この系列は自分で作ったので、理論上いちばん良い $\alpha$ も計算できます。水準のノイズと観測のノイズの分散の比 $q = \sigma_\eta^2 / \sigma_\varepsilon^2$ だけで決まります。

$$
\alpha = \frac{-q + \sqrt{q^2 + 4q}}{2}
$$

In [ ]:
tab_alpha <- data.frame(alpha = seq(0.01, 1, 0.01))
tab_alpha$sse <- sapply(tab_alpha$alpha, sse, y = sim$y)
alpha_hat <- optimize(sse, c(0, 1), y = sim$y)$minimum   # 2乗誤差の和が最小になる alpha
q <- 0.5^2 / 2^2                                         # ノイズの分散の比
alpha_theory <- (-q + sqrt(q^2 + 4 * q)) / 2
round(c(estimated = alpha_hat, theory = alpha_theory), 3)
ggplot(tab_alpha, aes(alpha, sse)) +
  geom_line() +
  geom_vline(xintercept = alpha_hat, colour = "blue") +
  geom_vline(xintercept = alpha_theory, colour = "red", linetype = "dashed") +
  labs(x = "alpha", y = "Sum of squared one-step errors (blue: estimated, red: theory)")

同じことを、`fable`パッケージの`ETS()`で行います。`error("A") + trend("N") + season("N")`が、単純指数平滑法の指定です。

In [ ]:
fit_ses <- sim |>
  model(ses = ETS(y ~ error("A") + trend("N") + season("N")))
tidy(fit_ses)   # alpha と、最初の水準 l[0]

`ETS()`は、最初の水準 $\ell_0$ も一緒に推定します。そのため、`alpha`は上の値と少し違います。

予測で確かめます。同じ作り方の系列を500本作り、最後の10時点を隠して、平均法・ナイーブ法・単純指数平滑法で当てます。

In [ ]:
set.seed(1)
one <- function(n = 200, h = 10) {
  mu <- 50 + cumsum(rnorm(n + h, sd = 0.5))
  y  <- mu + rnorm(n + h, sd = 2)
  train <- y[1:n]; test <- y[(n + 1):(n + h)]
  a <- optimize(sse, c(0, 1), y = train)$minimum
  fc <- c(mean = mean(train), naive = train[n], ses = ses(train, a)$level[n])
  c(sqrt(colMeans((outer(rep(1, h), fc) - test)^2)), alpha = a)   # 方法ごとのRMSEと、選ばれた alpha
}
res_ses <- rowMeans(replicate(500, one()))
round(res_ses, 3)

- 単純指数平滑法の誤差が、最も小さくなります。
- 選ばれた $\alpha$ の平均は、理論の値の近くにあります。

### A-2. 傾向と季節を足す

水準だけでなく、傾向（trend）と季節成分（seasonal component）も、同じやり方で更新できます。
状態（水準 $\ell_t$、傾き $b_t$、季節 $s_t$）を、予測の外れ $e_t = y_t - \hat{y}_t$ で少しずつ直します。

$$
\begin{aligned}
\hat{y}_t &= \ell_{t-1} + b_{t-1} + s_{t-m} \\
\ell_t &= \ell_{t-1} + b_{t-1} + \alpha\, e_t \\
b_t &= b_{t-1} + \beta\, e_t \\
s_t &= s_{t-m} + \gamma\, e_t
\end{aligned}
$$

$m$ は季節の周期（四半期なら4）です。この形のモデルを、ETS（Error, Trend, Seasonal）モデルと呼びます。

傾向と季節のある人工の系列（四半期、30年分）を作ります。

In [ ]:
set.seed(2)
nq <- 120
slope  <- 0.5 + cumsum(rnorm(nq, sd = 0.02))                  # 傾き：ゆっくり変わる
lev    <- 100 + cumsum(slope) + cumsum(rnorm(nq, sd = 0.5))   # 水準
season <- rep(c(8, -3, -10, 5), length.out = nq)              # 4期でくり返す上下（合計は0）
simq <- tsibble(Quarter = yearquarter("1990 Q1") + 0:(nq - 1),
                y = lev + season + rnorm(nq, sd = 2), index = Quarter)
ggplot(simq, aes(Quarter, y)) +
  geom_line() +
  labs(x = "Quarter", y = "Value")

最後の12期（3年）を隠し、3つのETSモデルと、第2回の3つのベンチマークで予測します。

- `ses`：水準だけ（単純指数平滑法）
- `holt`：水準 ＋ 傾向（ホルトの方法）
- `hw`：水準 ＋ 傾向 ＋ 季節（ホルト・ウィンターズの方法）

In [ ]:
train_q <- simq |> filter(Quarter <= max(Quarter) - 12)
fit_q <- train_q |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y),
        ses    = ETS(y ~ error("A") + trend("N") + season("N")),
        holt   = ETS(y ~ error("A") + trend("A") + season("N")),
        hw     = ETS(y ~ error("A") + trend("A") + season("A")))
fc_q <- fit_q |> forecast(h = 12)
autoplot(fc_q |> filter(.model %in% c("ses", "holt", "hw")),
         simq |> filter(Quarter >= yearquarter("2012 Q1")), level = NULL) +
  labs(x = "Quarter", y = "Value", colour = "Model")

In [ ]:
accuracy(fc_q, simq) |> select(.model, RMSE, MAE)
glance(fit_q) |> filter(.model %in% c("ses", "holt", "hw")) |> select(.model, AIC)

- `ses`は水平な線、`holt`は傾いた直線、`hw`は傾向と季節の両方を延ばします。
- `hw`の誤差が最も小さく、AICも最も小さくなります。

`hw`が推定した状態を見ます。`components()`は、各時点の水準・傾き・季節を返します。図を描く関数を用意します。

In [ ]:
# ETSモデルが推定した状態（水準・傾き・季節）と残差を、上から順に描く
plot_states <- function(fit) {
  cmp <- components(fit) |> as_tibble() |> filter(!is.na(y))   # 最初の状態だけの行を除く
  parts <- intersect(c("y", "level", "slope", "season", "remainder"), names(cmp))
  long <- do.call(rbind, lapply(parts, function(p)
    data.frame(Quarter = cmp$Quarter, part = p, value = cmp[[p]])))
  long$part <- factor(long$part, levels = parts)
  ggplot(long, aes(Quarter, value)) +
    geom_line() +
    facet_wrap(~part, ncol = 1, scales = "free_y") +
    labs(x = "Quarter", y = NULL)
}

In [ ]:
plot_states(fit_q |> select(hw))
tidy(fit_q |> select(hw)) |> filter(term %in% c("alpha", "beta", "gamma"))

第2回のSTL分解は、系列をあとから成分に分けました。ETSは、成分を1時点ずつ更新しながら進みます。

### A-3. ETSは統計モデルである

ETSモデルは、予測の外れ $e_t$ が、平均0・分散 $\sigma^2$ のiidな正規乱数だと仮定します。
この仮定から、予測区間（第4回）が出ます。単純指数平滑法では、$h$ 時点先の予測誤差の分散は次のとおりです。

$$
\sigma_h^2 = \sigma^2 \left\{ 1 + (h - 1)\,\alpha^2 \right\}
$$

A-1の系列で、`fable`が出す予測の分散と、この式を比べます。

In [ ]:
fc_ses <- fit_ses |> forecast(h = 10)
a  <- tidy(fit_ses)$estimate[1]      # alpha
s2 <- glance(fit_ses)$sigma2         # sigma^2
data.frame(h = 1:10,
           fable   = distributional::variance(fc_ses$y),
           formula = s2 * (1 + (0:9) * a^2)) |> round(3)
autoplot(fc_ses, sim |> filter(t > 150)) +
  labs(x = "Time t (step)", y = "Value")

2つの列は一致します。予測は水平な線ですが、区間は先へ行くほど広がります。

仮定が成り立っているかは、残差（1時点先の予測の外れ）で確かめます。A-2の`hw`の残差のACFと、リュング・ボックス検定（第4回）です。

In [ ]:
aug_q <- fit_q |> select(hw) |> augment()
aug_q |> ACF(.innov, lag_max = 16) |> autoplot() +
  labs(x = "Lag h (quarter)", y = "Sample ACF of residuals")
aug_q |> features(.innov, ljung_box, lag = 8)

残差のACFは、ほぼ破線の中にあります。リュング・ボックス検定のp値も大きく、残差に自己相関は見つかりません。

### A-4. 回帰の誤差が自己相関するとき

ここからは、ほかの系列を使います。第1回の見せかけの回帰では、無関係なランダムウォークどうしの回帰が、高い割合で有意になりました。
定常な系列でも、同じことが起きます。

$x_t$ と $y_t$ を、互いに無関係なAR(1)（$\phi = 0.8$）として作ります。長さは100です。回帰 $y_t = \beta_0 + \beta_1 x_t + \eta_t$ の本当の傾きは0です。

これを1000回くり返し、2つの方法で傾きを検定します。

- `ols`：いつもの回帰（誤差 $\eta_t$ がiidだと仮定する）
- `ar1`：誤差 $\eta_t$ がAR(1)に従うと仮定した回帰

In [ ]:
set.seed(1)
sim_ar1 <- function(n, phi) {           # AR(1)を1本作る（最初の100個は捨てる）
  x <- stats::filter(rnorm(n + 100), phi, method = "recursive")
  as.numeric(x[-(1:100)])
}
one_reg <- function(n = 100, phi = 0.8) {
  x <- sim_ar1(n, phi)
  y <- 1 + 0 * x + sim_ar1(n, phi)                       # 本当の傾きは0
  ols <- summary(lm(y ~ x))$coefficients[2, ]
  fit <- arima(y, order = c(1, 0, 0), xreg = cbind(x = x))   # 誤差がAR(1)の回帰
  b <- fit$coef[["x"]]; se <- sqrt(fit$var.coef["x", "x"])
  c(b_ols = ols[[1]], se_ols = ols[[2]], p_ols = ols[[4]],
    b_ar1 = b, se_ar1 = se, p_ar1 = 2 * pnorm(-abs(b / se)))
}
res_reg <- replicate(1000, one_reg())
reg_tab <- data.frame(
  method   = c("ols", "ar1"),
  rate     = c(mean(res_reg["p_ols", ] < 0.05), mean(res_reg["p_ar1", ] < 0.05)),   # 有意になった割合
  sd_b     = c(sd(res_reg["b_ols", ]), sd(res_reg["b_ar1", ])),                     # 傾きの推定値の、実際のばらつき
  mean_se  = c(mean(res_reg["se_ols", ]), mean(res_reg["se_ar1", ])))               # 出力された標準誤差の平均
reg_tab[, -1] <- round(reg_tab[, -1], 3)
reg_tab

- `ols`：有意になる割合が、5%よりずっと高くなります。出力された標準誤差（`mean_se`）が、実際のばらつき（`sd_b`）より小さいからです。
- `ar1`：割合は5%の近くに戻ります。標準誤差も、実際のばらつきとほぼ合います。

いつもの回帰の標準誤差が、どのくらい小さすぎるかは、式で分かります。$x_t$ と誤差 $\eta_t$ の自己相関を $\rho_x(h)$、$\rho_\eta(h)$ とすると、傾きの推定値の分散は、iidのときの式の

$$
1 + 2 \sum_{h=1}^{\infty} \rho_x(h)\,\rho_\eta(h)
$$

倍になります。第1回の標本平均の分散の倍率 $1 + 2\sum \rho(h)$ と同じ形です。どちらもAR(1)なら、倍率は $(1 + \phi_x \phi_\eta)/(1 - \phi_x \phi_\eta)$ です。

In [ ]:
phi <- 0.8
c(factor_theory = (1 + phi^2) / (1 - phi^2),
  factor_sim    = var(res_reg["b_ols", ]) / mean(res_reg["se_ols", ]^2))

### A-5. ARIMA誤差をもつ回帰

回帰の誤差 $\eta_t$ に、ARIMAモデルを当てはめます。これを、ARIMA誤差をもつ回帰（regression with ARIMA errors）と呼びます。

$$
y_t = \beta_0 + \beta_1 x_t + \eta_t, \qquad \eta_t \text{ はARIMAモデルに従う}
$$

今度は、本当に関係のある系列を作ります。$y_t = 2 + 1.5\,x_t + \eta_t$ で、$\eta_t$ はAR(1)（$\phi = 0.8$）です。

In [ ]:
set.seed(3)
nr <- 120
simr <- tsibble(t = 1:nr, x = sim_ar1(nr, 0.8), eta = sim_ar1(nr, 0.8), index = t) |>
  mutate(y = 2 + 1.5 * x + eta)
train_r <- simr |> filter(t <= 100)
fit_r <- train_r |>
  model(lm        = TSLM(y ~ x),                               # いつもの回帰
        arima     = ARIMA(y ~ pdq(1, 0, 0)),                   # 自分の過去だけ（AR(1)）
        reg_arima = ARIMA(y ~ x + pdq(1, 0, 0)))               # 誤差がAR(1)の回帰
tidy(fit_r) |> filter(term %in% c("x", "ar1")) |> select(.model, term, estimate, std.error)

- `lm`と`reg_arima`の傾きは、どちらも1.5の近くです。標準誤差は違います。
- `reg_arima`の`ar1`は、誤差の自己相関（本当は0.8）の推定値です。

残差を比べます。モデルの仮定は、最後に残るノイズ $w_t$ がホワイトノイズであることです。

In [ ]:
augment(fit_r) |>
  filter(.model != "arima") |>
  ACF(.innov, lag_max = 16) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF of residuals")

`lm`の残差には、強い自己相関が残っています。`reg_arima`の残差は、ほぼ破線の中です。

予測します。回帰で予測するには、先の時点の $x$ の値が要ります。ここでは、先の $x$ が分かっているものとして、その値を渡します。
モデルの外で決まり、入力として渡す変数を、外生変数（exogenous variable）と呼びます。

In [ ]:
test_r <- simr |> filter(t > 100)                  # 先の時点の x を含む
fc_r <- fit_r |> forecast(new_data = test_r)
autoplot(fc_r, simr |> filter(t > 60), level = NULL) +
  labs(x = "Time t (step)", y = "Value y", colour = "Model")
accuracy(fc_r, simr) |> select(.model, RMSE, MAE)

- `arima`（自分の過去だけ）は、すぐに平均へ戻ります。先の $x$ の動きを使えません。
- `lm`（$x$ だけ）は、$x$ の動きは追いますが、直前の誤差が続くことを使えません。
- `reg_arima`は両方を使うので、誤差が最も小さくなります。

## B. 実データを解析する

2つのデータを使います。どちらも四半期のデータです。

- B-1〜B-4：`tsibbledata`パッケージの`aus_production`（オーストラリアの生産量）。指数平滑法を使います。
- B-5〜B-9：`fpp3`パッケージの`us_change`（米国の経済指標の変化率）。回帰を使います。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
# --- B-1〜B-4：指数平滑法（オーストラリアの生産量）
prod_series <- "Beer"          # 系列："Beer" "Cement" "Electricity" "Gas" のどれか
prod_from   <- 1980            # 始まりの年：1956〜1980
# --- B-5〜B-9：回帰（米国の経済指標）
y_var       <- "Consumption"   # 説明される系列："Consumption" "Income" "Production" "Savings" "Unemployment"
x_var       <- "Income"        # 説明する系列：上の5つのうち、y_var と違うもの
year_from   <- 1970            # 始まりの年：1970〜1990
# --- 共通
h           <- 8               # 予測する四半期の数：4, 8, 12 のどれか

### B-1. データを確かめる

In [ ]:
glimpse(aus_production)
range(aus_production$Quarter)
c(rows = nrow(aus_production), gaps = has_gaps(aus_production)$.gaps)
colSums(is.na(as_tibble(aus_production)))

- 1行は1四半期です。1956年第1四半期から2010年第2四半期まで、218行あります。行の抜けはありません。
- ビール（`Beer`、百万リットル）、セメント（`Cement`、千トン）、電力（`Electricity`、ギガワット時）、ガス（`Gas`、ペタジュール）などの生産量です。
- `Tobacco`と`Bricks`には欠測があるので、今日は使いません。

### B-2. 指数平滑法：前処理

期間を絞り、設定で選んだ系列を`y`という名前にします。最後の`h`期をテストデータにします。

In [ ]:
prod <- aus_production |>
  filter(format(Quarter, "%Y") >= prod_from) |>
  mutate(y = .data[[prod_series]]) |>
  select(Quarter, y)
prod_lab <- c(Beer = "Beer production (megalitres)", Cement = "Cement production (thousand tonnes)",
              Electricity = "Electricity production (GWh)", Gas = "Gas production (petajoules)")[[prod_series]]
n_prod <- nrow(prod)
train_prod <- prod |> filter(Quarter <= max(Quarter) - h)
c(n = n_prod, train = nrow(train_prod), test = h, missing = sum(is.na(prod$y)))
ggplot(prod, aes(Quarter, y)) +
  geom_line() +
  labs(x = "Quarter", y = prod_lab)

### B-3. 指数平滑法：モデルと診断

3つのベンチマークと、3つのETSモデルを当てはめます。

- `ses`：水準だけ
- `hw`：水準 ＋ 傾向 ＋ 季節（すべて加法型）
- `auto`：`ETS()`に任せる。誤差・傾向・季節の型の組み合わせのうち、AICが最も小さいものを選ぶ

In [ ]:
fit_prod <- train_prod |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y),
        ses    = ETS(y ~ error("A") + trend("N") + season("N")),
        hw     = ETS(y ~ error("A") + trend("A") + season("A")),
        auto   = ETS(y, ic = "aic"))
ets_auto <- format(fit_prod$auto[[1]])       # 選ばれたモデルの名前
ets_auto
glance(fit_prod) |> filter(.model %in% c("ses", "hw", "auto")) |> select(.model, sigma2, AIC)
tidy(fit_prod) |> filter(.model == "hw", term %in% c("alpha", "beta", "gamma")) |> select(.model, term, estimate)

`hw`が推定した状態（水準・傾き・季節）です。

In [ ]:
plot_states(fit_prod |> select(hw))

残差（1時点先の予測の外れ）に、自己相関が残っていないかを確かめます。

In [ ]:
aug_prod <- fit_prod |> select(hw, auto) |> augment()
aug_prod |> ACF(.innov, lag_max = 16) |> autoplot() +
  labs(x = "Lag h (quarter)", y = "Sample ACF of residuals")
lb_prod <- aug_prod |> features(.innov, ljung_box, lag = 8)
lb_prod

### B-4. 指数平滑法：予測を比べる

最後の`h`期を予測し、第2回と同じ方法で比べます。

In [ ]:
fc_prod <- fit_prod |> forecast(h = h)
autoplot(fc_prod |> filter(.model %in% c("snaive", "hw", "auto")),
         prod |> filter(Quarter > max(Quarter) - 5 * h), level = NULL) +
  labs(x = "Quarter", y = prod_lab, colour = "Model")
acc_prod <- accuracy(fc_prod, prod) |> select(.model, RMSE, MAE)
acc_prod

時系列交差検証でも比べます。訓練データを`h`期ずつ伸ばしながら、6回くり返します。

In [ ]:
cv_prod <- train_prod |>
  stretch_tsibble(.init = n_prod - 6 * h, .step = h) |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y),
        ses    = ETS(y ~ error("A") + trend("N") + season("N")),
        hw     = ETS(y ~ error("A") + trend("A") + season("A")),
        auto   = ETS(y, ic = "aic")) |>
  forecast(h = h) |>
  accuracy(prod) |>
  select(.model, RMSE, MAE)
cv_prod

読み方：

- B-3：`auto`が選ぶ型は、系列によって変わります。`M`は乗法型（multiplicative）で、季節の上下の幅が水準に比例して大きくなる系列で選ばれます。傾向の`Ad`は、傾きが先へ行くほど小さくなる型（damped）です。
- B-3：残差のACFが破線の外に出るラグがあれば、モデルが取りこぼした動きが残っています。リュング・ボックス検定のp値と合わせて読みます。
- B-4：AICで選んだモデルが、テストデータでも最も良いとは限りません。季節ナイーブ法に勝てないこともあります。

### B-5. 回帰：データと前処理

`us_change`は`fpp3`パッケージに入っています。パッケージ全体は読み込まず、データだけを取り出します。

In [ ]:
us_change <- fpp3::us_change
glimpse(us_change)
range(us_change$Quarter)
c(rows = nrow(us_change), missing = sum(is.na(us_change)), gaps = has_gaps(us_change)$.gaps)

- 1行は1四半期です。1970年第1四半期から2019年第2四半期まで、198行あります。欠測も、行の抜けもありません。
- 個人消費（`Consumption`）、可処分所得（`Income`）、生産（`Production`）、貯蓄（`Savings`）の、前の期からの変化率（%）と、失業率の変化（`Unemployment`）です。

`us_change`から、説明される系列を`y`、説明する系列を`x`という名前にします。最後の`h`期をテストデータにします。

In [ ]:
us <- us_change |>
  filter(format(Quarter, "%Y") >= year_from) |>
  mutate(y = .data[[y_var]], x = .data[[x_var]]) |>
  select(Quarter, y, x)
unit <- function(v) if (v == "Unemployment") "change" else "% change"
y_lab <- sprintf("%s (%s)", y_var, unit(y_var))
x_lab <- sprintf("%s (%s)", x_var, unit(x_var))
n_us <- nrow(us)
train_us <- us |> filter(Quarter <= max(Quarter) - h)
test_us  <- us |> filter(Quarter >  max(Quarter) - h)
c(n = n_us, train = nrow(train_us), test = nrow(test_us))

In [ ]:
us_long <- rbind(data.frame(Quarter = us$Quarter, series = y_lab, value = us$y),
                 data.frame(Quarter = us$Quarter, series = x_lab, value = us$x))
us_long$series <- factor(us_long$series, levels = c(y_lab, x_lab))
ggplot(us_long, aes(Quarter, value)) +
  geom_line() +
  facet_wrap(~series, ncol = 1, scales = "free_y") +
  labs(x = "Quarter", y = NULL)

この系列は、もとの量（消費額など）の変化率です。変化率は、もとの量の差分にあたります。第4回の「差分をとって定常に直す」処理が、すでに済んでいます。図に、季節の上下は見えません。

### B-6. 回帰：いつもの回帰と、その残差

まず、いつもの回帰（誤差がiidだと仮定する）を当てはめます。

In [ ]:
fit_lm <- train_us |> model(lm = TSLM(y ~ x))
tidy(fit_lm) |> select(term, estimate, std.error, statistic, p.value)

残差のACFと、リュング・ボックス検定です。

In [ ]:
aug_lm <- augment(fit_lm)
aug_lm |> ACF(.innov, lag_max = 16) |> autoplot() +
  labs(x = "Lag h (quarter)", y = "Sample ACF of residuals (lm)")
lb_lm <- aug_lm |> features(.innov, ljung_box, lag = 8)
lb_lm

A-4の倍率 $1 + 2\sum_h \rho_x(h)\,\rho_\eta(h)$ を、標本ACFから見積もります（ラグ8まで）。1より大きければ、いつもの回帰の標準誤差は小さすぎます。

In [ ]:
rho_x   <- acf(train_us$x, lag.max = 8, plot = FALSE)$acf[-1]        # x の標本ACF
rho_eta <- acf(aug_lm$.innov, lag.max = 8, plot = FALSE)$acf[-1]     # 残差の標本ACF
round(rbind(rho_x, rho_eta), 2)
factor_hat <- 1 + 2 * sum(rho_x * rho_eta)
factor_hat

### B-7. 回帰：ARIMA誤差をもつ回帰

誤差にARMAモデルを当てはめます。次数は、第4回と同じく、`ARIMA()`がAICcの小さいものを選びます。
系列はすでに変化率で、季節の上下も見えないので、差分はとらず（`pdq(d = 0)`）、季節の項も入れません（`PDQ(0, 0, 0)`）。

In [ ]:
fit_ra <- train_us |> model(reg_arima = ARIMA(y ~ x + pdq(d = 0) + PDQ(0, 0, 0)))
ra_name <- format(fit_ra$reg_arima[[1]])     # 選ばれた誤差のモデル
ra_name
coef_ra <- tidy(fit_ra) |> select(term, estimate, std.error, statistic, p.value)
coef_ra

In [ ]:
aug_ra <- augment(fit_ra)
aug_ra |> ACF(.innov, lag_max = 16) |> autoplot() +
  labs(x = "Lag h (quarter)", y = "Sample ACF of residuals (reg_arima)")
dof <- sum(grepl("^(ar|ma)[0-9]", coef_ra$term))       # 誤差のモデルの係数の数
lb_ra <- aug_ra |> features(.innov, ljung_box, lag = 8, dof = dof)
lb_ra

`x`の係数と標準誤差を、2つのモデルで比べます。

In [ ]:
coef_x <- rbind(cbind(model = "lm",        tidy(fit_lm) |> filter(term == "x") |> select(estimate, std.error, p.value)),
                cbind(model = "reg_arima", coef_ra |> filter(term == "x") |> select(estimate, std.error, p.value)))
coef_x

### B-8. 回帰：予測を比べる

6つのモデルで、最後の`h`期を予測します。

- 自分の過去だけ：`mean`、`naive`、`ets`、`arima`
- ほかの系列だけ：`lm`
- 両方：`reg_arima`

`lm`と`reg_arima`には、テストデータの期間の`x`の値（実際に起きた値）を渡します。

In [ ]:
fit_us <- train_us |>
  model(mean      = MEAN(y),
        naive     = NAIVE(y),
        ets       = ETS(y ~ error("A") + trend("N") + season("N")),
        arima     = ARIMA(y ~ pdq(d = 0) + PDQ(0, 0, 0)),
        lm        = TSLM(y ~ x),
        reg_arima = ARIMA(y ~ x + pdq(d = 0) + PDQ(0, 0, 0)))
fc_us <- fit_us |> forecast(new_data = test_us)
autoplot(fc_us |> filter(.model %in% c("ets", "arima", "reg_arima")),
         us |> filter(Quarter > max(Quarter) - 5 * h), level = NULL) +
  labs(x = "Quarter", y = y_lab, colour = "Model")
acc_us <- accuracy(fc_us, us) |> select(.model, RMSE, MAE)
acc_us

実際の予測では、先の`x`は分かりません。先の`x`を、訓練データの`x`の平均で置き換えた場合と比べます。

In [ ]:
test_guess <- test_us |> mutate(x = mean(train_us$x))      # 先の x を知らないとき
fc_guess <- fit_us |> select(lm, reg_arima) |> forecast(new_data = test_guess)
acc_guess <- accuracy(fc_guess, us) |> select(.model, RMSE, MAE)
acc_guess

### B-9. 回帰：時系列交差検証

訓練データを`h`期ずつ伸ばしながら、6回くり返します。`lm`と`reg_arima`には、毎回、予測する期間の`x`の実際の値を渡します。

In [ ]:
cv_us <- train_us |> stretch_tsibble(.init = n_us - 6 * h, .step = h)
future_us <- new_data(cv_us, h) |>
  left_join(as_tibble(us) |> select(Quarter, x), by = "Quarter")   # 予測する期間の x
acc_cv_us <- cv_us |>
  model(mean      = MEAN(y),
        naive     = NAIVE(y),
        ets       = ETS(y ~ error("A") + trend("N") + season("N")),
        arima     = ARIMA(y ~ pdq(d = 0) + PDQ(0, 0, 0)),
        lm        = TSLM(y ~ x),
        reg_arima = ARIMA(y ~ x + pdq(d = 0) + PDQ(0, 0, 0))) |>
  forecast(new_data = future_us) |>
  accuracy(us) |>
  select(.model, RMSE, MAE)
acc_cv_us

読み方：

- B-6：残差のACFが破線の外に出ていれば、いつもの回帰の標準誤差とp値は信用できません。
- B-7：誤差にARMAモデルを入れると、係数と標準誤差が変わります。残差の自己相関が消えたかを、ACFとリュング・ボックス検定で確かめます。
- B-8：`lm`と`reg_arima`は、先の`x`を知っているぶん有利です。先の`x`を知らないとき（`acc_guess`）の誤差と比べてください。
- B-9：1回の評価（B-8）と、6回の評価で、順位が変わることがあります。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `prod_series` | `"Beer"` `"Cement"` `"Electricity"` `"Gas"` |
| `prod_from` | 1956〜1980 |
| `y_var` | `"Consumption"` `"Income"` `"Production"` `"Savings"` `"Unemployment"` |
| `x_var` | 上の5つのうち、`y_var`と違うもの |
| `year_from` | 1970〜1990 |
| `h` | 4、8、12 |

読むときの観点：

1. 系列（`prod_series`）を替えると、`auto`が選ぶETSモデルの型はどう変わるか。図の季節の上下の幅と合わせて読む。
2. 始まりの年（`prod_from`）を早くすると、`hw`の残差のACFと、予測の順位はどう変わるか。長い期間を使うほど良い、と言えるか。
3. `y_var`と`x_var`の組を替えると、いつもの回帰の残差の自己相関は、どの組で強いか。そのとき、`x`の係数の標準誤差はどちらへ動くか。
4. `reg_arima`が`arima`に勝つのは、どの組か。先の`x`を知らないとき（`acc_guess`）でも勝つか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Beer、1980年から、Consumption と Income、1970年から、8期先）の結果と比べてください。